# Photon Budget Calculations



This section will estimate the amount of photons produced by the Cherenkov radiation, and the amount of usable photons. The source to be used is defined as:

0.1 µci Strontium-90 (Accuracy ±20%)

This is equivalent to **3700** disintegrations per second (d/s). The first step to this process is calculating the number of photons per unit track length, which will be done via the Frank-Tamm formula. The formula is as follows: 

$\frac{dN}{dx} = 2\pi\alpha \cdot \sin^2\theta_c \cdot \left(\frac{1}{\lambda_1} - \frac{1}{\lambda_2}\right)$

Where $\frac{dN}{dx}$ represents the number of photons per unit track length, $\alpha$ is a constant equal to 1/137, $\sin^2\theta_c$ is the angle of the Cherenkov radiation previously determined, and ${\lambda_1}$ and ${\lambda_2}$ are the edges of the wavelength band. For this project, the values will be defined as 400nm and 550nm due to it being where the Cherenkov spectrum is the brighest and the chosen camera equipment having a strong quantum efficiency. (ZWO ASI533MM Pro)

To determine $\sin^2\theta_c$, we will use the formula:

$\sin^2\theta_c = 1 - \frac{1}{\beta^2 n^2}$

Where n represents the refractive index of the medium the Cherenkov radiation occurs in, and $\beta$ represents the speed of the particle over the speed of light. ($\frac{v}{c}$) It will be defined as 1 for the theoretical optimal case for now.




In [4]:
#code to calculate the frank-tamm formula
import math
#sin^2(theta) calculation function
def calculate_angle(beta, n):
    return 1 - (1 / (beta**2 * n**2))

#constants
alpha_coeff_constant = 1/137 * 2 * math.pi

def frank_tamm_formula(beta, n, lambda1, lambda2):
    sin_squared_theta = calculate_angle(beta, n)
    if sin_squared_theta < 0:
        return 0  # No Cherenkov radiation if sin^2(theta) is negative, edge case
    else:
        return alpha_coeff_constant * sin_squared_theta * (1/lambda1 - 1/lambda2)

print("n = 1.33, beta = 1, dN/dx is calulcated to be:", frank_tamm_formula(1, 1.33, 4.0e-5, 5.5e-5))


n = 1.33, beta = 1, dN/dx is calulcated to be: 135.9234716485893


The peak rate of photons emitted has now been determined to be ~136 photons/cm. In real conditions, this will never occur as due to special relativity $\beta$ can never be equal to or greater than 1. In order to calculate the true number of photons, $\beta$ must be completely substituted out of the function.

The first step to this process is to determine the total number of photons emitted per decay, as this is the ultimate goal of this process. To do this, dN/dx must be integrated over the path length X. This results in the function:

$N = \int \frac{dN}{dx} dx$

Now, the original Frank-Tamm formula is substituted back in:

$N = \int 2\pi\alpha \sin^2\theta_c \left( \frac{1}{\lambda_1} - \frac{1}{\lambda_2} \right) dx$

The function now returns the total number of photons per decay. The next step is to figure out how to eliminate $\beta$. To achieve this, the $\sin^2\theta_c$ must be isolated in the integral. Because the other terms in the integrals are all constants, they may be factored out via the constant multiple rule. This results in the equation:

$N =  2\pi\alpha\left( \frac{1}{\lambda_1} - \frac{1}{\lambda_2} \right) \int \sin^2\theta_c (x)  dx$

To remove $\beta$ from this equation, it will first be substitued for the lorentz term, $\gamma$. This is done via the equation:

$\beta = \sqrt{1 - \frac{1}{\gamma^2}}$

Now, the lorentz term is related to the kinetic energy of the particle via the equation:

$\gamma = 1 + \frac{m_e c^2}{E}$

In this equation, $m_e$ represents the electrons rest mass. That multiplied by $c^2$ is defined as 0.511 MeV. Now, it has been determined that $\sin^2\theta_c$ can be computed in terms of E, not relying on $\beta$ anymore. As E changes over time, it is defined as the function:

$S = -\dfrac{dE}{dx}$

Rearranging in terms of dx in order to substitue, the equation becomes:

$dx = -\dfrac{dE}{S}$

Thus, it is substitued back into the original equation, resulting in: 

$N =  2\pi\alpha\left( \frac{1}{\lambda_1} - \frac{1}{\lambda_2} \right) \int \sin^2\theta_c (E) (-\frac{dE}{S}) dx$

Simplifying, the result becomes:

$N = 2\pi\alpha \left( \frac{1}{\lambda_1} - \frac{1}{\lambda_2} \right) \int_{E_{th}}^{E_0} \frac{{\sin^2\theta_c}(E)}{S} dE$

The variable S, stopping power will be represented by the approximation determined by NIST ESTAR as ~1.9 $\frac{MeV}{cm}$. Now that S is a constant, it comes out of the integral as well. This results in the final equation to be:

$N = 2\pi\alpha \left( \frac{1}{\lambda_1} - \frac{1}{\lambda_2} \right) \frac{1}{S} \int_{E_{th}}^{E_0} \sin^2\theta_c E dE$














In [7]:
#code to compute the Frank-Tamm integral
import math
#constants
alpha_coeff_constant = 1/137 * 2 * math.pi
wavelength_constant = 1/(4.0e-5) - 1/(5.5e-5)
stopping_power_constant = 1 / 1.9
front_term = alpha_coeff_constant * wavelength_constant * stopping_power_constant
def sin_2_theta_c(E):
    n = 1.33
    m_e = 0.511
    gamma = 1 + E/m_e          # kinetic energy → gamma
    beta_squared = 1 - 1/gamma**2
    beta = math.sqrt(beta_squared)
    return calculate_angle(beta, n)

def frank_tamm_integral(E, steps):
    E_threshold = 0.264   # MeV, Cherenkov threshold in water
    if E <= E_threshold:
        return 0          # no emission below threshold
    integral_value = 0
    for i in range(steps):  # integrate in specified number of steps
        E_step = E_threshold + (E - E_threshold) * (i / steps)
        sin2_theta = sin_2_theta_c(E_step)
        if sin2_theta < 0:
            sin2_theta = 0  # no emission if sin^2(theta) is negative
        integral_value += sin2_theta * (E - E_threshold) / steps
    return integral_value

print("2.28 MeV:", front_term * frank_tamm_integral(2.28, 1000))
print("1.5 MeV:", front_term * frank_tamm_integral(1.5, 1000))
print("0.93 MeV:", front_term * frank_tamm_integral(0.93, 1000))



2.28 MeV: 115.31806818115355
1.5 MeV: 63.09414802633124
0.93 MeV: 27.611493936544694


The results end up as:

# 2.28 MeV = ~115.3 Photons/electron
# 1.5 MeV = ~63.1 Photons/electron
# 0.93 MeV = ~27.6 Photons/electron 